In [1]:
# importing libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


# 1.Mount the google drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


# Step 1: Data Ingestion & Transformation

In [3]:
df = pd.read_csv('/content/drive/MyDrive/uk_household_costs (1) 1(in).csv')
display(df.head(100))

,Date,Region,Household_ID,Monthly_Income,Energy_Bill,Rent_Mortgage,Food_Spend,Savings_Buffer,Household_Type,Receiving_Support
0,24/06/2024,South East,UK-HH-5849,3588.88,166.05,1470.00,508.48,168.50,Detached,No
1,19/05/2024,Wales,UK-HH-4646,2428.81,72.62,3956.60,371.28,193.77,Terraced,No
2,7/9/2024,London,UK-HH-2244,4605.61,113.73,1900.46,517.48,1509.49,Detached,Yes
3,21/10/2024,North West,UK-HH-1690,3645.92,118.52,1084.50,383.88,1760.08,Semi-Detached,No
4,14/08/2024,North West,UK-HH-1545,2590.58,128.88,1223.40,523.60,2376.70,Terraced,No
...,...,...,...,...,...,...,...,...,...,...
95,20/04/2024,South East,UK-HH-3002,2624.81,155.59,1184.16,332.70,7680.09,Semi-Detached,No
96,18/08/2024,South East,UK-HH-1687,3837.23,58.31,1595.79,236.91,1580.49,Terraced,No
97,28/02/2024,South East,UK-HH-8184,5072.36,97.54,1715.91,530.23,1621.06,Terraced,No
98,4/5/2024,Wales,UK-HH-1749,3398.98,139.72,867.68,384.21,47.68,Flat,No


In [4]:
df.describe()

,Monthly_Income,Energy_Bill,Rent_Mortgage,Food_Spend,Savings_Buffer
count,2000.000000,2000.000000,2000.000000,2000.00000,2000.000000
mean,3265.946055,145.069050,1387.679905,452.17626,3821.496815
std,1034.898355,40.022397,1407.872484,118.09496,3845.020838
min,1000.000000,50.000000,400.000000,150.00000,5.970000
25%,2537.155000,116.857500,798.750000,369.74000,1150.610000
50%,3147.630000,144.765000,1036.395000,453.23500,2650.465000
75%,3878.305000,173.250000,1410.350000,531.27000,5370.032500
max,8152.690000,270.430000,15773.490000,822.96000,37794.850000


In [5]:
df.isnull()

,Date,Region,Household_ID,Monthly_Income,Energy_Bill,Rent_Mortgage,Food_Spend,Savings_Buffer,Household_Type,Receiving_Support
0,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
1995,False,False,False,False,False,False,False,False,False,False
1996,False,False,False,False,False,False,False,False,False,False
1997,False,False,False,False,False,False,False,False,False,False
1998,False,False,False,False,False,False,False,False,False,False


In [6]:
df.isnull().sum()

,0
Date,0
Region,0
Household_ID,0
Monthly_Income,0
Energy_Bill,0
Rent_Mortgage,0
Food_Spend,0
Savings_Buffer,0
Household_Type,0
Receiving_Support,0


In [7]:
df.Date.head(10)

,Date
0,24/06/2024
1,19/05/2024
2,7/9/2024
3,21/10/2024
4,14/08/2024
5,5/7/2024
6,24/02/2024
7,3/4/2024
8,23/01/2024
9,27/11/2024


In [8]:
# STEP 3: Fix date format
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

/tmp/ipykernel_3089/3059010745.py:2: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")


In [9]:
df['Date'].head(10)

,Date
0,2024-06-24
1,2024-05-19
2,2024-09-07
3,2024-10-21
4,2024-08-14
5,2024-07-05
6,2024-02-24
7,2024-04-03
8,2024-01-23
9,2024-11-27


In [10]:
df["Monthly_Income"].head(10)

,Monthly_Income
0,3588.88
1,2428.81
2,4605.61
3,3645.92
4,2590.58
5,1884.84
6,2914.13
7,4776.27
8,3010.39
9,4675.79


# Step 2: Heuristic Anomaly Filtering (The "Financial Reality" Filter)

In [11]:
# STEP 4: Convert money columns to numbers
money_cols = [
    "Monthly_Income",
    "Energy_Bill",
    "Rent_Mortgage",
    "Food_Spend",
    "Savings_Buffer"
]

for col in money_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [12]:
df["Monthly_Income"].head(10)

,Monthly_Income
0,3588.88
1,2428.81
2,4605.61
3,3645.92
4,2590.58
5,1884.84
6,2914.13
7,4776.27
8,3010.39
9,4675.79


In [13]:
df.dtypes

,0
Date,datetime64[ns]
Region,object
Household_ID,object
Monthly_Income,float64
Energy_Bill,float64
Rent_Mortgage,float64
Food_Spend,float64
Savings_Buffer,float64
Household_Type,object
Receiving_Support,object


# Step 3: Feature Engineering

In [14]:
# STEP 5: Detect bad/corrupted data

df["Essential_Spend"] = (
    df["Rent_Mortgage"] +
    df["Energy_Bill"] +
    df["Food_Spend"]
)

bad_data = df[
    df["Essential_Spend"] > df["Monthly_Income"] * 1.5
]

clean_data = df[
    df["Essential_Spend"] <= df["Monthly_Income"] * 1.5
]

In [15]:
# STEP 6: Save bad data for audit
bad_data.to_csv("quarantined_data.csv", index=False)

In [16]:
print("Bad rows:", len(bad_data))
print("Clean rows:", len(clean_data))

Bad rows: 115
Clean rows: 1885


# Step 4: Non-ML AI Integration (Rule-Based Expert System) *

In [17]:
# STEP 7: Create new financial features

clean_data["Disposable_Income"] = (
    clean_data["Monthly_Income"]
    - clean_data["Rent_Mortgage"]
    - clean_data["Energy_Bill"]
    - clean_data["Food_Spend"]
)

clean_data["Housing_Ratio"] = (
    clean_data["Rent_Mortgage"] / clean_data["Monthly_Income"]
) * 100

clean_data["Energy_Ratio"] = (
    clean_data["Energy_Bill"] / clean_data["Monthly_Income"]
) * 100

/tmp/ipykernel_3089/2377059277.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_data["Disposable_Income"] = (
/tmp/ipykernel_3089/2377059277.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_data["Housing_Ratio"] = (
/tmp/ipykernel_3089/2377059277.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_gu

In [18]:
print(type(clean_data))

<class 'pandas.core.frame.DataFrame'>


In [19]:
print("Clean rows:", len(clean_data))

Clean rows: 1885


In [20]:
# STEP 8: Create rule-based AI expert system

def classify_household(row):
    if (
        row["Housing_Ratio"] > 40
        and row["Energy_Ratio"] > 15
        and row["Savings_Buffer"] < 500
    ):
        return "Critical"

    elif row["Disposable_Income"] < 0:
        return "High Risk"

    elif (
        row["Housing_Ratio"] > 30
        or row["Energy_Ratio"] > 10
        or row["Savings_Buffer"] < 1000
    ):
        return "Vulnerable"

    else:
        return "Stable"

In [21]:
print(classify_household(clean_data.iloc[7]))

Stable


In [22]:
clean_data["Vulnerability_Tier"] = clean_data.apply(
    classify_household,
    axis=1
)

/tmp/ipykernel_3089/2134026351.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_data["Vulnerability_Tier"] = clean_data.apply(


In [23]:
print(clean_data["Vulnerability_Tier"].value_counts())

Vulnerability_Tier
Vulnerable    1236
Stable         591
High Risk       57
Critical         1
Name: count, dtype: int64


In [24]:
# STEP 9: Apply expert system to every row
clean_data["Vulnerability_Tier"] = clean_data.apply(
    classify_household,
    axis=1
)

/tmp/ipykernel_3089/2175186178.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_data["Vulnerability_Tier"] = clean_data.apply(


In [25]:
# STEP 10: Save final cleaned dataset
clean_data.to_csv("cleaned_ai_enriched_household_costs.csv", index=False)

In [26]:
# Download file from Colab
from google.colab import files
files.download("cleaned_ai_enriched_household_costs.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Step 5: Database Loading

In [27]:
!pip install sqlalchemy pymysql

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 2.1 MB/s eta 0:00:00


In [38]:
from cryptography.fernet import Fernet

# IMPORTANT: In a real application, you should generate a key once and store it securely.
# DO NOT regenerate the key every time you run your script, or you won't be able to decrypt old data.
# For demonstration purposes, we generate a new key here.
key = Fernet.generate_key()
cipher = Fernet(key)

# Original username and password
original_username = b'your_actual_username'
original_password = b'your_actual_password'

# Encrypt the values
enc_user = cipher.encrypt(original_username)
enc_pass = cipher.encrypt(original_password)

# Now, to use them later, you would load your stored key and then decrypt:
# (assuming 'key' is loaded from a secure storage)
# cipher = Fernet(key)

username = cipher.decrypt(enc_user).decode()
password = cipher.decrypt(enc_pass).decode()

print("Generated Key:", key.decode())
print("Encrypted Username:", enc_user)
print("Encrypted Password:", enc_pass)
print("Decrypted Username:", username)
print("Decrypted Password:", password)

# Setting global variables 'username' and 'password' for subsequent use, if needed.
# Make sure to replace 'your_actual_username' and 'your_actual_password' with your real credentials.
# For connecting to the database, you would use these decrypted values:
# username = username # already assigned above
# password = password # already assigned above


Generated Key: swpCcuEjj36BtXNuNxnY3LWQZq-qwirUxKA3b90W9is=
Encrypted Username: b'gAAAAABp70AtwCIzr3E_gcbAj7ynrWYtl8hjEM-Jf7Cau7Q9uHhxi0nkCvze3MSwt7IumqL6SgcLxoGmCAgpmINS0xJERoHA2vFErq6UbPCq3pYpdzmlfoI='
Encrypted Password: b'gAAAAABp70AtIRtG3gzJijfYG2VCe8xn3lUqNWvXc-WJv2vIoXSdf4dw7hZVnHW-OqStAnhikMFiO2BakVPDi7neR3ir-ONLwwb33hRWFvVQzxKwn33ISwg='
Decrypted Username: your_actual_username
Decrypted Password: your_actual_password


In [39]:
host = "YOUR_REMOTE_MYSQL_HOST"